In [36]:
import pandas as pd
import numpy as np

# file_path = r"C:\Users\AGFirass\Documents\GitHub\Transformer-Based-DDoS-Detection\notebooks\feature_importance\correlation_analysis\reduced_dataset\reduced_dataset_correlation_filtered.parquet"

file_path = "C:/Users/AGFirass/Documents/GitHub/Transformer-Based-DDoS-Detection/data/subset/01-12/flow-level/combined_subset_50k.parquet"
df = pd.read_parquet(file_path)
df.columns = df.columns.str.replace(" ", "", regex=False)

In [37]:
print("="*80)
print("DATASET OVERVIEW")
print("="*80)
print(f"Shape: {df.shape}")
print("\nColumns:\n", df.columns.tolist())
print("\nData Types:\n", df.dtypes)

DATASET OVERVIEW
Shape: (600000, 88)

Columns:
 ['Unnamed:0', 'FlowID', 'SourceIP', 'SourcePort', 'DestinationIP', 'DestinationPort', 'Protocol', 'Timestamp', 'FlowDuration', 'TotalFwdPackets', 'TotalBackwardPackets', 'TotalLengthofFwdPackets', 'TotalLengthofBwdPackets', 'FwdPacketLengthMax', 'FwdPacketLengthMin', 'FwdPacketLengthMean', 'FwdPacketLengthStd', 'BwdPacketLengthMax', 'BwdPacketLengthMin', 'BwdPacketLengthMean', 'BwdPacketLengthStd', 'FlowBytes/s', 'FlowPackets/s', 'FlowIATMean', 'FlowIATStd', 'FlowIATMax', 'FlowIATMin', 'FwdIATTotal', 'FwdIATMean', 'FwdIATStd', 'FwdIATMax', 'FwdIATMin', 'BwdIATTotal', 'BwdIATMean', 'BwdIATStd', 'BwdIATMax', 'BwdIATMin', 'FwdPSHFlags', 'BwdPSHFlags', 'FwdURGFlags', 'BwdURGFlags', 'FwdHeaderLength', 'BwdHeaderLength', 'FwdPackets/s', 'BwdPackets/s', 'MinPacketLength', 'MaxPacketLength', 'PacketLengthMean', 'PacketLengthStd', 'PacketLengthVariance', 'FINFlagCount', 'SYNFlagCount', 'RSTFlagCount', 'PSHFlagCount', 'ACKFlagCount', 'URGFlagCount'

In [38]:
print("\nMissing values per column:\n", df.isna().sum())


Missing values per column:
 Unnamed:0        0
FlowID           0
SourceIP         0
SourcePort       0
DestinationIP    0
                ..
IdleMax          0
IdleMin          0
SimillarHTTP     0
Inbound          0
Label            0
Length: 88, dtype: int64


In [39]:
print("\nChecking for infinite values...")
inf_cols = df.columns[(df == np.inf).any() | (df == -np.inf).any()]
print("Infinite values found in columns:", list(inf_cols))


Checking for infinite values...
Infinite values found in columns: ['FlowBytes/s', 'FlowPackets/s']


In [40]:
df = df.replace([np.inf, -np.inf], np.nan)

In [41]:
# Fill NaNs with median (numeric cols only)
for col in df.select_dtypes(include=[np.number]).columns:
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val)

# Fill NaNs in non-numeric columns with mode
for col in df.select_dtypes(exclude=[np.number]).columns:
    mode_val = df[col].mode()[0] if not df[col].mode().empty else None
    df[col] = df[col].fillna(mode_val)

# Missing values after
print("\nMissing values after handling:\n", df.isna().sum())


Missing values after handling:
 Unnamed:0        0
FlowID           0
SourceIP         0
SourcePort       0
DestinationIP    0
                ..
IdleMax          0
IdleMin          0
SimillarHTTP     0
Inbound          0
Label            0
Length: 88, dtype: int64


In [42]:
df.shape

(600000, 88)

In [43]:
# 7. Check 'Label' column
print("\nLabel column unique values:")
print(df["Label"].unique())

if pd.api.types.is_numeric_dtype(df["Label"]):
    print("Label is numeric — possible label encoding.")
else:
    print("Label is not numeric — probably categorical.")


Label column unique values:
['BENIGN' 'DrDoS_DNS' 'DrDoS_LDAP' 'DrDoS_MSSQL' 'DrDoS_NetBIOS'
 'DrDoS_NTP' 'DrDoS_SNMP' 'DrDoS_SSDP' 'DrDoS_UDP' 'Syn' 'TFTP' 'UDP-lag']
Label is not numeric — probably categorical.


In [53]:
df = df.drop(columns=["FlowID"], errors='ignore')

In [45]:
from sklearn.preprocessing import LabelEncoder

# Encode SimillarHTTP
if df["SimillarHTTP"].dtype == object:
    df["SimillarHTTP"] = df["SimillarHTTP"].fillna("Unknown")
    df["SimillarHTTP"] = LabelEncoder().fit_transform(df["SimillarHTTP"])

In [46]:
# Convert Timestamp to datetime
df["Timestamp"] = pd.to_datetime(df["Timestamp"], errors='coerce')

In [47]:
# Convert Timestamp to numeric (UNIX timestamp in seconds)
df["Timestamp"] = df["Timestamp"].astype("int64") // 10**9  # seconds

In [48]:
# Encode Label (0–11)
label_encoder = LabelEncoder()
df["Label"] = label_encoder.fit_transform(df["Label"])

In [49]:
print("\nLabel classes mapping:")
for idx, cls in enumerate(label_encoder.classes_):
    print(f"{cls} -> {idx}")


Label classes mapping:
BENIGN -> 0
DrDoS_DNS -> 1
DrDoS_LDAP -> 2
DrDoS_MSSQL -> 3
DrDoS_NTP -> 4
DrDoS_NetBIOS -> 5
DrDoS_SNMP -> 6
DrDoS_SSDP -> 7
DrDoS_UDP -> 8
Syn -> 9
TFTP -> 10
UDP-lag -> 11


In [50]:
# Encode Source IP and Destination IP into integer IDs
for col in ["SourceIP", "DestinationIP"]:
    df[col] = LabelEncoder().fit_transform(df[col].astype(str))

In [54]:
# Check datatypes
print("\nData Types after processing:\n", df.dtypes)


Data Types after processing:
 Unnamed:0            int64
SourceIP             int64
SourcePort           int64
DestinationIP        int64
DestinationPort      int64
                    ...   
IdleMax            float64
IdleMin            float64
SimillarHTTP         int64
Inbound              int64
Label                int64
Length: 87, dtype: object


In [55]:
print("\nFinal dataset shape:", df.shape)


Final dataset shape: (600000, 87)


In [56]:
df.to_parquet("data_50k.parquet", index=False)

In [57]:
df.columns

Index(['Unnamed:0', 'SourceIP', 'SourcePort', 'DestinationIP',
       'DestinationPort', 'Protocol', 'Timestamp', 'FlowDuration',
       'TotalFwdPackets', 'TotalBackwardPackets', 'TotalLengthofFwdPackets',
       'TotalLengthofBwdPackets', 'FwdPacketLengthMax', 'FwdPacketLengthMin',
       'FwdPacketLengthMean', 'FwdPacketLengthStd', 'BwdPacketLengthMax',
       'BwdPacketLengthMin', 'BwdPacketLengthMean', 'BwdPacketLengthStd',
       'FlowBytes/s', 'FlowPackets/s', 'FlowIATMean', 'FlowIATStd',
       'FlowIATMax', 'FlowIATMin', 'FwdIATTotal', 'FwdIATMean', 'FwdIATStd',
       'FwdIATMax', 'FwdIATMin', 'BwdIATTotal', 'BwdIATMean', 'BwdIATStd',
       'BwdIATMax', 'BwdIATMin', 'FwdPSHFlags', 'BwdPSHFlags', 'FwdURGFlags',
       'BwdURGFlags', 'FwdHeaderLength', 'BwdHeaderLength', 'FwdPackets/s',
       'BwdPackets/s', 'MinPacketLength', 'MaxPacketLength',
       'PacketLengthMean', 'PacketLengthStd', 'PacketLengthVariance',
       'FINFlagCount', 'SYNFlagCount', 'RSTFlagCount', 'PSHFla